In [ ]:
import matplotlib.pyplot as plt
import scipy.sparse as sparse
import scipy.sparse.linalg as sla
import numpy as np
import shapely
import shapely.geometry
from descartes import PolygonPatch
%matplotlib inline

prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']

import warnings
warnings.filterwarnings("ignore", message="The array interface is deprecated and will no longer work in Shapely")

In [ ]:
def oneDpoisson(N, RHS_func):
    # Given number of intervals on [0,1], discretize
    # -u'' = RHS_func(x) using centred FD discretization

    # Compute widths of elements
    h = 1.0 / N

    # Compute entries
    subd_entries = -(1.0 / h**2) * np.ones(N - 1)
    diag_entries = (2.0 / h**2) * np.ones(N - 1)
    supd_entries = -(1.0 / h**2) * np.ones(N - 1)

    # Create matrix
    A = sparse.diags([subd_entries[1:], diag_entries,
                      supd_entries[:-1]], [-1, 0, 1], format="csr")

    # Create RHS, no adjustment needed at endpoints
    x = np.arange(h, 1.0 - 0.5 * h, h)
    b = RHS_func(x)

    return A, b

def twoDpoisson(Nx, Ny, RHS_func):
    # Given number of intervals on [0,1]^2, discretize
    # -\Delta u = RHS_func(x,y) using centred FD discretization

    # Use one-D routine to create matrix parts, with fake RHS
    def zero_func(x):
        return 0.0 * x
    Ax, tmp1 = oneDpoisson(Nx, zero_func)
    Ay, tmp2 = oneDpoisson(Ny, zero_func)

    # Now assemble
    A = sparse.kron(Ax, sparse.identity(Ny - 1, format="csr")) + \
        sparse.kron(sparse.identity(Nx - 1, format="csr"), Ay)

    # Set up arrays for efficient evaluation of RHS
    x1d = np.arange(1.0 / Nx, 1.0 - 0.5 / Nx, 1.0 / Nx)
    y1d = np.arange(1.0 / Ny, 1.0 - 0.5 / Ny, 1.0 / Ny)
    x, y = np.meshgrid(x1d, y1d)

    # fancy evaluation, with option to flatten to match matrix ordering
    b = RHS_func(x.flatten('F'), y.flatten('F'))

    return A, b, x, y

def TwoD_nonoverlap(N, s):
    # Compute and return index sets for dividing N x N grid into s x s
    # set of subdomains, assuming N/s is an integer

    # Subdomain size in 1D, assuming it is an integer
    subd_size = int(N / s)

    # Initialize counters and memory
    index_array = np.zeros((subd_size**2, s**2), dtype=int)
    subd_count = 0
    one_vec = np.ones(subd_size, dtype=int)

    # Loop over subdomains
    for j in np.arange(s):
        for i in np.arange(s):
            # Compute start and endpoints of 2D index ranges
            i_start = i * subd_size
            i_end = (i + 1) * subd_size
            j_start = j * subd_size
            j_end = (j + 1) * subd_size

            # Create vectors of indices in range
            i_vec = np.arange(i_start, i_end)
            j_vec = np.arange(j_start, j_end)

            # Now compute lexicographic indices from 1D indices
            index_array[:, subd_count] = np.kron(
                one_vec, i_vec) + N * np.kron(j_vec, one_vec)
            # Increment column counter
            subd_count = subd_count + 1

    return index_array

def Schwarz_restriction(N, index_array):
    # Given an index array for an N x N mesh, determine R0, accounting
    # for overlap

    # Allocate R0 in coordinate format, scaling
    (size_subs, num_subs) = np.shape(index_array)
    R0_data = np.ones(num_subs * size_subs)
    R0_row_ind = np.zeros(num_subs * size_subs, dtype='int')
    R0_col_ind = np.zeros(num_subs * size_subs, dtype='int')
    scaling = np.zeros(N**2)

    # Loop over indices, count for scaling, initialize R0
    count = 0
    for i in np.arange(size_subs):
        for j in np.arange(num_subs):
            scaling[index_array[i, j]] += 1
            R0_row_ind[count] = j
            R0_col_ind[count] = index_array[i, j]
            count += 1

    # Form R0 as CSR
    R0 = sparse.csr_matrix(
        (R0_data, (R0_row_ind, R0_col_ind)), shape=(
            num_subs, N**2))
    # Now scale columns of R0
    D = sparse.diags([1.0 / scaling], [0], format="csr")
    R0 = R0 @ D

    return R0, D

def TwoD_minimaloverlap(N, s):
    # Compute and return index sets for dividing N x N grid into s x s
    # set of subdomains with minimal overlap, which assumes (N-1)/s is
    # an integer

    # Subdomain size in 1D, assuming it is an integer
    subd_size = int((N - 1) / s)

    # Initialize counters and memory
    index_array = np.zeros(((subd_size + 1)**2, s**2), dtype=int)
    subd_count = 0
    one_vec = np.ones(subd_size + 1, dtype=int)

    # Loop over subdomains
    for j in np.arange(s):
        for i in np.arange(s):
            # Compute start and endpoints of 2D index ranges
            i_start = i * subd_size
            i_end = (i + 1) * subd_size
            j_start = j * subd_size
            j_end = (j + 1) * subd_size

            # Create vectors of indices in range
            i_vec = np.arange(i_start, i_end + 1)
            j_vec = np.arange(j_start, j_end + 1)

            # Now compute lexicographic indices from 1D indices
            index_array[:, subd_count] = np.kron(
                one_vec, i_vec) + N * np.kron(j_vec, one_vec)
            # Increment column counter
            subd_count = subd_count + 1

    return index_array

def additive_Schwarz(A, r, index_array):
    # Given A and r, compute z = Mr, where M is the additive Schwarz
    # preconditioner on A with non-overlapping subdomains given by
    # index_array

    # zero z
    z = np.zeros_like(r)
    # Loops over subdomains and solve
    for i in np.arange(np.shape(index_array)[1]):
        inds = index_array[:, i]
        z[inds] = z[inds] + sla.spsolve(A[inds[:, None], inds], r[inds])

    return z

def additive_Schwarz_2level(A, r, R0, Ac, index_array):
    # Given A and r, compute z = Mr, where M is the additive Schwarz
    # preconditioner on A with non-overlapping subdomains given by
    # index_array

    # zero z
    z = np.zeros_like(r)
    # Loops over subdomains and solve
    for i in np.arange(np.shape(index_array)[1]):
        inds = index_array[:, i]
        z[inds] = z[inds] + sla.spsolve(A[inds[:, None], inds], r[inds])

    # Coarse level
    z = z + R0.transpose() * sla.spsolve(Ac, R0 @ r)

    return z

In [ ]:
def RHS(x,y):
    return 2*(x-x**2+y-y**2)

N = 32
A, b, x, y = twoDpoisson(N+1,N+1,RHS)
x = x.ravel()
y = y.ravel()
h = x[1] - x[0]

index_set = TwoD_nonoverlap(N, 2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for I, color in zip(index_set.T, colors):
    xi, yi  = x[I], y[I]
    xyi = np.vstack((xi, yi)).T
    xyimin = xyi.min(axis=0)
    xyimax = xyi.max(axis=0)
    
    box = shapely.geometry.box(xyimin[0], xyimin[1], xyimax[0], xyimax[1])
    patch = PolygonPatch(box, fc=color, ec=color, alpha=0.5, zorder=-1)
    ax.plot(xi, yi, 'ko')
    ax.add_patch(patch)
ax.axis('square')

xi, yi  = x[I], y[I]
xyi = np.vstack((xi, yi)).T

In [ ]:
uex = sla.spsolve(A, b)
plt.pcolormesh(x.reshape(N, N), y.reshape(N, N), uex.reshape(N, N), shading='auto')

In [ ]:
sub_size = 16
index_set = TwoD_nonoverlap(N,sub_size)
R0,D = Schwarz_restriction(N,index_set)
Ac = R0@(A@R0.transpose())
def prec_apply(r):
    #return additive_Schwarz_2level(A,r,R0,Ac,index_set)
    return additive_Schwarz(A, r, index_set)
M_2level = sla.LinearOperator((N**2,N**2),matvec=prec_apply)

In [ ]:
np.random.seed(234786)
u0 = np.random.rand(M_2level.shape[0])
u = u0.copy()
res = []
for i in range(100):
    u[:] = u + 0.5 * M_2level @ (b - A @ u)
    resnorm = np.linalg.norm(b - A @ u)
    res.append(resnorm)
    print(resnorm)

In [ ]:
uex = sla.spsolve(A, b)
plt.pcolormesh(x.reshape(N, N), y.reshape(N, N), u.reshape(N, N), shading='auto')

In [ ]:
np.linalg.norm(b - A @ u)

In [ ]:
pres = []
def cb(x):
    pres.append(np.linalg.norm(b - A * x))
def cb2(r):
    pres.append(r)
sla.iterative.gmres(A, b, M=M_2level, callback=cb2, callback_type='pr_norm', tol=1e-12)

In [ ]:
pres

In [ ]:
res = np.array(res) / res[0]
pres = np.array(pres) / pres[0]
plt.semilogy(res, label='DD')
plt.semilogy(pres, label='GMRES+DD')
plt.legend()